# GTSRB frequency-domain robustness: FreqDebias-AT vs DAT — Colab runner v2

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells top to bottom.

**What changed from v1**
* No dependency conflicts: only AutoAttack is installed (`--no-deps`); everything else is Colab's preinstalled version.
* The dataset is the official 39,209-image GTSRB training set (v1 used torchvision's 26,640-image archive, which caused the `AssertionError`).
* Every step runs from the repository folder and stops the notebook with a clear message if it fails.
* Training writes to the fast local disk (`/content/out`); results are synced to **persistent storage** every 10 minutes and at the end of each step.

**Persistent storage** (cell 4 picks automatically, `STORAGE = "auto"`; it stops with instructions if neither works):
1. **Hugging Face (preferred)** — a private dataset repository under your free Hugging Face account. One-time setup: create an account at huggingface.co → Settings → Access Tokens → *New token*, type **Write** → in Colab click the 🔑 *Secrets* icon in the left bar → add a secret named `HF_TOKEN` with the token, and switch *Notebook access* on.
2. **Google Drive (fallback)** — used automatically if there is no `HF_TOKEN` secret or Hugging Face fails.

**After a disconnect:** reconnect, run cells 1–5 again (they restore your results from storage), then re-run the step you were on. Training resumes from its last checkpoint.

## 1. Settings

In [ ]:
import os, sys, json, subprocess, time

STORAGE = "auto"          # "auto" (Hugging Face if an HF_TOKEN secret exists, else Google Drive), "hf", "drive", "none"
HF_REPO_NAME = "gtsrb-freq-robust-runs"                   # private dataset repo created under your HF account
DRIVE_DIR = "/content/drive/MyDrive/gtsrb_freq_robust"    # used only by the Drive fallback
REPO_URL = "https://github.com/NeuralNomadic/-DAT-and-FreqDebias.git"
BRANCH = "main"
SYNC_INTERVAL_MIN = 10    # upload new results at most every N minutes during training (plus at the end of each step)

# Local test switch used by the repository's own notebook test. Leave RUNNER_TEST unset on Colab.
TEST_MODE = os.environ.get("RUNNER_TEST") == "1"
ROOT = os.environ.get("RUNNER_ROOT", "/content") if TEST_MODE else "/content"
if TEST_MODE:
    REPO_URL = os.environ["RUNNER_REPO_URL"]
    STORAGE = os.environ.get("RUNNER_STORAGE", "none")
SMOKE = ["--smoke"] if TEST_MODE else []

REPO, DATA, OUT = f"{ROOT}/repo", f"{ROOT}/data", f"{ROOT}/out"
os.makedirs(ROOT, exist_ok=True)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"   # child processes print UTF-8 (tables contain ∞, ±)


def run(*cmd, cwd=None, env_extra=None, check=True):
    """Run a command from the repository folder, stream its output live, stop the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    env = os.environ.copy()
    env.update(env_extra or {})
    p = subprocess.Popen(cmd, cwd=cwd or REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    while True:
        chunk = p.stdout.read1(4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"step failed (exit code {rc}): {' '.join(cmd)}  -- see the output above")
    return None if check else rc   # None: nothing is echoed under the cell


def py(module, *args, **kw):
    return run(sys.executable, "-m", module, *args, **kw)


print("repository:", REPO_URL, "| local results folder:", OUT, "| test mode:", TEST_MODE)

## 2. GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
elif TEST_MODE:
    print("TEST_MODE: no GPU, running on CPU")
else:
    raise RuntimeError("NO GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then run the notebook again.")

## 3. Get the code and install AutoAttack
The copy in `/content/repo` is always reset to the latest GitHub version (edit code on GitHub, not here).

In [ ]:
if os.path.isdir(f"{REPO}/.git"):
    run("git", "-C", REPO, "fetch", "--quiet", "origin", BRANCH, cwd=ROOT)
    run("git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}", cwd=ROOT)
else:
    run("git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO, cwd=ROOT)
run("git", "log", "-1", "--format=code version: %h %s")
# Only AutoAttack is installed; --no-deps guarantees no preinstalled Colab package is touched.
run(sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-r", "requirements-colab.txt")
run(sys.executable, "-c", "import torch, torchvision, numpy, scipy, sklearn, pandas, matplotlib, yaml, tqdm, PIL, "
    "huggingface_hub, autoattack; print('all imports ok | torch', torch.__version__, '| torchvision', "
    "torchvision.__version__, '| numpy', numpy.__version__)")

## 4. Persistent storage
Restores earlier results (logs, metrics, data caches). Checkpoints are fetched only when a step needs them.

In [ ]:
def _secret(name):
    """Read a Colab secret (fresh every time, so an edited secret is picked up), else an environment variable."""
    value = None
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        pass
    value = value or os.environ.get(name)
    return value.strip().strip('"').strip("'").strip() if value else None


HF_HELP = """How to fix the Hugging Face token:
  1. huggingface.co -> your picture (top right) -> Settings -> Access Tokens -> Create new token.
  2. Token type: Write. Create it and copy the value immediately (it is shown only once; it starts with hf_).
  3. Colab: key icon in the left bar (Secrets) -> HF_TOKEN -> paste the new value as it is, no spaces or quotes.
     Keep 'Notebook access' switched on.
  4. Run this cell again."""

DRIVE_HELP = """How to fix Google Drive:
  Run this cell again and finish the Google sign-in popup: choose the same Google account you use for Colab and
  tick EVERY permission checkbox (Colab refuses the mount if any is left unticked). If no popup appears, allow
  pop-ups for colab.research.google.com."""


def try_hf():
    token = _secret("HF_TOKEN")
    if not token:
        print("No HF_TOKEN secret found.\n" + HF_HELP)
        return None
    if not token.startswith("hf_"):
        print("The HF_TOKEN secret does not look like a Hugging Face token (it should start with hf_).")
    try:
        from huggingface_hub import HfApi
        api = HfApi(token=token)
        user = api.whoami()["name"]
        repo_id = f"{user}/{HF_REPO_NAME}"
        api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "Invalid user token" in msg:
            reason = "Hugging Face rejected the token (it is wrong, incomplete, or was deleted)."
        elif "403" in msg:
            reason = "The token is valid but cannot create repositories: it must be of type Write."
        else:
            reason = f"{type(e).__name__}: {msg[:300]}"
        print(f"Hugging Face storage is not usable. {reason}\n" + HF_HELP)
        return None
    os.environ["HF_TOKEN"] = token
    print(f"Hugging Face storage: https://huggingface.co/datasets/{repo_id} (private)")
    return f"hf:{repo_id}"


def try_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
        os.makedirs(DRIVE_DIR, exist_ok=True)
    except Exception as e:
        print(f"Google Drive is not usable: {type(e).__name__}: {str(e)[:300]}\n" + DRIVE_HELP)
        return None
    print("Google Drive storage:", DRIVE_DIR)
    return f"dir:{DRIVE_DIR}"


def setup_storage(choice):
    if choice.startswith("dir:"):
        return choice
    if choice == "none":
        print("\033[91mSTORAGE = 'none': results live only on this VM and are lost when it disconnects.\033[0m")
        return "none"
    spec = try_hf() if choice in ("auto", "hf") else None
    if spec is None and choice in ("auto", "drive"):
        if choice == "auto":
            print("\nTrying Google Drive instead.")
        spec = try_drive()
    if spec is None:
        raise RuntimeError("No persistent storage is available, so results would be lost when Colab disconnects. "
                           "Fix one of the problems printed above and run this cell again (or set STORAGE = 'none' "
                           "in cell 1 to continue without storage; not recommended).")
    return spec


os.environ["OUT_SYNC"] = setup_storage(STORAGE)
os.environ["OUT_SYNC_INTERVAL"] = str(SYNC_INTERVAL_MIN * 60)
os.makedirs(OUT, exist_ok=True)
py("src.storage", "pull", "--out", OUT)

## 5. Data: official GTSRB, track split, 32×32 caches
First time: downloads ~370 MB and decodes 51,839 images (about 5 minutes). Later sessions reuse the caches from storage.

In [ ]:
py("src.make_split", "--data", DATA, "--out", OUT)

## 6. Helpers (run once per session)

In [ ]:
def train_eval(exp, seed=0, evaluate=True):
    """Train (resumes automatically), evaluate, push results. Safe to re-run after a disconnect."""
    py("src.train", "--config", f"configs/{exp}.yaml", "--seed", seed, "--data", DATA, "--out", OUT, "--resume", *SMOKE)
    if evaluate:
        py("src.evaluate", "--exp", exp, "--seed", seed, "--data", DATA, "--out", OUT, *SMOKE)
    py("src.storage", "push", "--out", OUT)
    show(exp, seed)


def show(exp, seed=0):
    p = f"{OUT}/runs/{exp}/seed{seed}/metrics.json"
    if not os.path.exists(p):
        print(f"{exp} seed {seed}: no metrics yet")
        return
    m = json.load(open(p))
    keys = ["clean", "fgsm", "pgd20", "pgd100", "cw_inf", "autoattack", "corruption_mean", "best_epoch", "sec_per_epoch"]
    print(f"{exp} seed {seed}: " + ", ".join(f"{k}={m.get(k)}" for k in keys))
    bad = [k for k, v in (m.get("sanity") or {}).items() if not v]
    print("sanity checks:", "\033[91mFAILED " + ", ".join(bad) + "\033[0m" if bad else "passed")


def status():
    """One line per run: training progress and headline metrics."""
    import csv
    runs = sorted(os.path.join(dp) for dp, dn, fn in os.walk(f"{OUT}/runs") if "train_log.csv" in fn)
    if not runs:
        print("no runs yet")
    for d in runs:
        rows = list(csv.DictReader(open(f"{d}/train_log.csv")))
        m = json.load(open(f"{d}/metrics.json")) if os.path.exists(f"{d}/metrics.json") else {}
        name = os.path.relpath(d, f"{OUT}/runs")
        last = rows[-1] if rows else {}
        print(f"{name:24s} epochs {len(rows):3d} {'DONE' if os.path.exists(f'{d}/DONE') else 'running/partial':16s}"
              f" val_pgd20 {last.get('val_pgd20_acc', '') or '-':>8}  test clean {m.get('clean', '-')}  "
              f"pgd20 {m.get('pgd20', '-')}  AA {m.get('autoattack', '-')}")


def show_image(path):
    from IPython.display import Image, display
    if os.path.exists(path):
        display(Image(filename=path))
    else:
        print("not found:", path)


status()

## 7. Optional self-test (~3 minutes, CPU)
All unit tests plus a 2-batch smoke run of the four Tier-1 methods. Uses a separate scratch folder and never touches your results or storage. Recommended once after the code changes.

In [ ]:
RUN_SELF_TEST = False
if RUN_SELF_TEST:
    no_sync = {"OUT_SYNC": "none"}
    run(sys.executable, "-m", "pytest", "-q", "-x", "-W", "ignore", "tests/test_data.py", "tests/test_fourier_fomixup.py",
        "tests/test_model_attacks.py", "tests/test_storage.py", "tests/test_corruptions_and_loader.py", env_extra=no_sync)
    for exp in ["E0_standard", "E3_base", "E5_fd_at", "E6_dat"]:
        py("src.train", "--config", f"configs/{exp}.yaml", "--data", f"{ROOT}/no_data", "--out", f"{ROOT}/selftest",
           "--smoke", env_extra=no_sync)
    print("self-test passed")

## 8. Tier 1 — the core comparison (E0, E3, E5, E6; seed 0)
Run one cell at a time. **Milestone 1 check after E0:** test clean accuracy well above 95 %, PGD-20 near 0 %.

In [ ]:
train_eval("E0_standard")

In [ ]:
# Milestone 2: E3 (shared adversarial base). Note sec_per_epoch to plan the remaining tiers.
train_eval("E3_base")

In [ ]:
# Milestone 3a: Fo-Mixup band masks from E0 (needed by E4, E5, A1-A4) + visual check: signs must stay readable.
py("src.precompute_fomixup", "--data", DATA, "--out", OUT, *SMOKE)
show_image(f"{OUT}/figures/fomixup_grid.png")

In [ ]:
train_eval("E5_fd_at")

In [ ]:
# Milestone 4: DAT. The grid is drawn with the trained generator after E6 finishes.
train_eval("E6_dat")
py("src.make_grids", "--data", DATA, "--out", OUT, "--which", "dat", *SMOKE)
show_image(f"{OUT}/figures/dat_grid.png")

In [ ]:
# Milestone 5: tables and figures from everything evaluated so far.
py("src.aggregate", "--out", OUT)
p = f"{OUT}/results/table_main.md"
if os.path.exists(p):
    from IPython.display import Markdown, display
    display(Markdown(open(p, encoding="utf-8").read()))

In [ ]:
# Environment record for the thesis (run once, after the first successful run).
os.makedirs(f"{OUT}/env", exist_ok=True)
with open(f"{OUT}/env/requirements.lock.txt", "w") as f:
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
py("src.storage", "push", "--out", OUT)

## Tier 2 — baselines and H2 (E1, E2, E4; seed 0)

In [ ]:
for exp in ["E1_pgdat", "E2_trades", "E4_fd_clean"]:
    train_eval(exp)

## Tier 3 — error bars for the core comparison (seeds 1, 2 of E3, E5, E6)

In [ ]:
for exp in ["E3_base", "E5_fd_at", "E6_dat"]:
    for seed in [1, 2]:
        train_eval(exp, seed)

## Tier 4 — ablations A1–A6 (seed 0)

In [ ]:
for exp in ["A1", "A2", "A3", "A4", "A5", "A6"]:
    train_eval(exp)

## Tier 5 — error bars for the baselines (seeds 1, 2 of E1, E2)

In [ ]:
for exp in ["E1_pgdat", "E2_trades"]:
    for seed in [1, 2]:
        train_eval(exp, seed)

## Status and final push (run any time)

In [ ]:
status()
py("src.storage", "status", "--out", OUT)
py("src.storage", "push", "--out", OUT)